In [16]:
import chromadb

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [18]:
# 1. EL DOCUMENTO
texto = """
El esquema de vacunación para perros de raza mediana inicia entre las 6 y 8 semanas de vida.
Las vacunas esenciales incluyen Parvovirus, Moquillo, Hepatitis Infecciosa, Leptospirosis y Rabia.
Se recomienda la vacuna contra la Tos de las Perreras (Bordetella) para perros con alta socialización en parques.
Es obligatorio realizar una desparasitación interna y externa antes de aplicar cualquier refuerzo inmunológico.
Los perros adultos requieren un refuerzo anual de la vacuna polivalente y la rabia para mantener la inmunidad.
"""

#### 1. Chunking (segmentación)

In [4]:
# 2. VISUALIZANDO EL CHUNKING
print("--- PASO 1: CHUNKING ---")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10)
chunks = text_splitter.split_text(texto)
for i, chunk in enumerate(chunks):
    print(f"Pedazo {i}: [{chunk}]")

--- PASO 1: CHUNKING ---
Pedazo 0: [El esquema de vacunación para perros de raza mediana inicia entre las 6 y 8]
Pedazo 1: [las 6 y 8 semanas de vida.]
Pedazo 2: [Las vacunas esenciales incluyen Parvovirus, Moquillo, Hepatitis Infecciosa,]
Pedazo 3: [Leptospirosis y Rabia.]
Pedazo 4: [Se recomienda la vacuna contra la Tos de las Perreras (Bordetella) para perros]
Pedazo 5: [perros con alta socialización en parques.]
Pedazo 6: [Es obligatorio realizar una desparasitación interna y externa antes de aplicar]
Pedazo 7: [aplicar cualquier refuerzo inmunológico.]
Pedazo 8: [Los perros adultos requieren un refuerzo anual de la vacuna polivalente y la]
Pedazo 9: [y la rabia para mantener la inmunidad.]


#### 2. Embedding (incrustación)

In [5]:
# 3. VISUALIZANDO EL EMBEDDING
print("\n--- PASO 2: EMBEDDINGS (Vectores) ---")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
ejemplo_vector = model.encode(chunks[0])
print(f"Texto: '{chunks[0]}'")
print(f"Se convirtió en un vector de {len(ejemplo_vector)} números.")
print(f"Primeros 5 números del vector: {ejemplo_vector[:5]} ...")


--- PASO 2: EMBEDDINGS (Vectores) ---
Texto: 'El esquema de vacunación para perros de raza mediana inicia entre las 6 y 8'
Se convirtió en un vector de 384 números.
Primeros 5 números del vector: [ 0.1895457   0.1615695  -0.5933467   0.18739508 -0.12281191] ...


In [6]:
# 4. GUARDANDO EN CHROMADB
print("\n--- PASO 3: GUARDANDO EN LA BASE DE DATOS ---")
db_path = "data/chroma_db"

client = chromadb.PersistentClient(path=db_path)
collection = client.get_or_create_collection(name="vacunas_mascotas")
for i, chunk in enumerate(chunks):
    vector = model.encode(chunk).tolist()
    collection.add(ids=[f"id_{i}"], embeddings=[vector], documents=[chunk])
print("¡Todo guardado y listo para buscar!")


--- PASO 3: GUARDANDO EN LA BASE DE DATOS ---
¡Todo guardado y listo para buscar!


In [7]:
col = client.get_collection("vacunas_mascotas")
docs = col.get(include=["documents", "embeddings"])

for i, (doc, emb) in enumerate(zip(docs["documents"], docs["embeddings"]), start=1):
    print(f"Documento {i}:")
    print(f"Vector: {emb[:5]} ...")
    print(doc)
    print()

Documento 1:
Vector: [ 0.18954571  0.16156951 -0.59334671  0.18739508 -0.12281191] ...
El esquema de vacunación para perros de raza mediana inicia entre las 6 y 8

Documento 2:
Vector: [ 0.16368811 -0.00612855 -0.0072543   0.09171524 -0.07419303] ...
las 6 y 8 semanas de vida.

Documento 3:
Vector: [-0.13082345  0.13814792 -0.53200561 -0.19667046  0.20610684] ...
Las vacunas esenciales incluyen Parvovirus, Moquillo, Hepatitis Infecciosa,

Documento 4:
Vector: [-0.23635451 -0.14844893 -0.26187149  0.11576502  0.10745367] ...
Leptospirosis y Rabia.

Documento 5:
Vector: [-0.18009377 -0.09756526 -0.35891959  0.08697904  0.07631765] ...
Se recomienda la vacuna contra la Tos de las Perreras (Bordetella) para perros

Documento 6:
Vector: [ 0.58421022 -0.12272245 -0.28506494  0.23437309 -0.13635533] ...
perros con alta socialización en parques.

Documento 7:
Vector: [ 0.08223484  0.10976413 -0.08026934 -0.16739023  0.04758089] ...
Es obligatorio realizar una desparasitación interna y externa 

#### 3. Retrieval (Recuperación)

In [8]:
# 5. BUSCANDO Y VIENDO RESULTADOS
print("\n--- PASO 4: BÚSQUEDA ---")
query = "¿Qué tipo de vacuna sería la adecuada para un perro de raza mediana?"
print(f"Pregunta del usuario: '{query}'")


--- PASO 4: BÚSQUEDA ---
Pregunta del usuario: '¿Qué tipo de vacuna sería la adecuada para un perro de raza mediana?'


In [9]:
query_vec = model.encode(query).tolist()
resultados = collection.query(query_embeddings=[query_vec], n_results=2)

In [10]:
print("\nResultados encontrados por orden de relevancia:")
for i in range(len(resultados['documents'][0])):
    doc = resultados['documents'][0][i]
    distancia = resultados['distances'][0][i]
    print(f"Ranking {i+1}: {doc} (Distancia: {distancia:.4f})")


Resultados encontrados por orden de relevancia:
Ranking 1: Se recomienda la vacuna contra la Tos de las Perreras (Bordetella) para perros (Distancia: 6.1408)
Ranking 2: Los perros adultos requieren un refuerzo anual de la vacuna polivalente y la (Distancia: 7.8826)


#### 4. Generación 
Ejemplo: Creando un agente con langchain 
[https://docs.langchain.com/oss/python/langchain/rag ]

In [11]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.chat_models import ChatOllama
from typing import List


# Inicializar componentes
embeddings = SentenceTransformerEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = Chroma(
    persist_directory="data/chroma_db",
    collection_name="vacunas_mascotas",
    embedding_function=embeddings
)


@tool(response_format="content_and_artifact")
def retrieve_context(query: str) -> str:
    """
    Recupera información relevante para ayudar a responder una consulta.
    
    Args:
        query: Consulta del usuario
        
    Returns:
        Contexto serializado de los documentos recuperados
    """
    retrieved_docs = vectorstore.similarity_search(query, k=3)
    serialized = "\n\n".join(
        (f"Fuente: {doc.metadata}\nContenido: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized

C:\Users\DELL\AppData\Local\Temp\ipykernel_25184\3658916887.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(


In [12]:
# Prueba rápida de conexión
from langchain_community.chat_models import ChatOllama

model = ChatOllama(model="gemma3:1b")
response = model.invoke("Hola")
print(response.content)

C:\Users\DELL\AppData\Local\Temp\ipykernel_25184\1552239610.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  model = ChatOllama(model="gemma3:1b")


Hola! ¿En qué puedo ayudarte hoy? 😊



In [13]:
from langchain_chroma import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.chat_models import ChatOllama
from typing import List
import traceback


# Inicializar componentes
embeddings = SentenceTransformerEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = Chroma(
    persist_directory="data/chroma_db",
    collection_name="vacunas_mascotas",
    embedding_function=embeddings
)

In [14]:
class SimpleRAGAgent:
    """
    Agente simple de RAG usando LangChain con Ollama (versión simplificada).
    """
    
    def __init__(
        self,
        ollama_model: str = "gemma3:1b",
        k: int = 3
    ):
        """
        Inicializa el agente RAG.
        """
        # Inicializar modelo Ollama
        self.llm = ChatOllama(
            model=ollama_model,
            base_url="http://localhost:11434",
            temperature=0.3,
            timeout=180
        )
        
        self.k = k
    
    def retrieve(self, query: str) -> List[str]:
        """
        Recupera documentos relevantes.
        """
        docs = vectorstore.similarity_search(query, k=self.k)
        return [doc.page_content for doc in docs]
    
    def generate(self, query: str) -> str:
        """
        Genera una respuesta usando el modelo directamente con el contexto.
        """
        try:
            # Retrieval
            docs = self.retrieve(query)
            
            if not docs:
                return "No se encontraron documentos relevantes."
            
            # Construir contexto
            context = "\n".join([f"- {doc}" for doc in docs])
            
            # Crear prompt simple
            prompt = f"""Eres un asistente experto en vacunación de mascotas. 
            Responde la pregunta del usuario basándote SOLO en la siguiente información:

            INFORMACIÓN:
            {context}

            PREGUNTA: {query}

            RESPUESTA:"""
            
            # Generar respuesta
            response = self.llm.invoke(prompt)
            
            # Extraer contenido
            if hasattr(response, 'content'):
                return response.content
            elif isinstance(response, dict):
                return response.get('content', str(response))
            else:
                return str(response)
                
        except Exception as e:
            error_details = traceback.format_exc()
            return f"Error al generar respuesta:\n{str(e)}\n\nDetalles:\n{error_details}"

In [15]:
# Crear y probar el agente
agent = SimpleRAGAgent(ollama_model="gemma3:1b")

query = "¿Qué tipo de vacuna sería la adecuada para un perro de raza mediana?"

print("=== RETRIEVAL ===")
docs = agent.retrieve(query)
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc}")

print("\n=== GENERACIÓN ===")
respuesta = agent.generate(query)
print(respuesta)

=== RETRIEVAL ===
1. Se recomienda la vacuna contra la Tos de las Perreras (Bordetella) para perros
2. Los perros adultos requieren un refuerzo anual de la vacuna polivalente y la
3. El esquema de vacunación para perros de raza mediana inicia entre las 6 y 8

=== GENERACIÓN ===
La vacuna adecuada para un perro de raza mediana es la vacuna contra la Tos de las Perreras (Bordetella).

